# Watcher - Performance Analysis in ComCam On-Sky Campaign

This analysis is part of the preparation for the **LSSTCam On-Sky Workshop**. The goal is to evaluate the performance of the **Watcher** during the **ComCam On-Sky campaign**, which took place from **October 24, 2024, to December 11, 2024**. 

This notebook focuses on a specific subtask: **Watcher - Mute Duration**. We need to study the duration of muted alarms using a histogram.

In [1]:
from astropy.time import Time

from lsst.sitcom.vandv.logger import create_logger
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats


## Information about the Watcher

In [2]:
# Create an EFD client instance
client = makeEfdClient()

In [3]:
# make a list of all topics in the EFD related to Watcher
topics = await client.get_topics()
for topic in topics:
    if 'Watcher' in topic:
        print(topic)

lsst.sal.Watcher.ackcmd
lsst.sal.Watcher.command_acknowledge
lsst.sal.Watcher.command_disable
lsst.sal.Watcher.command_enable
lsst.sal.Watcher.command_exitControl
lsst.sal.Watcher.command_makeLogEntry
lsst.sal.Watcher.command_mute
lsst.sal.Watcher.command_setLogLevel
lsst.sal.Watcher.command_showAlarms
lsst.sal.Watcher.command_standby
lsst.sal.Watcher.command_start
lsst.sal.Watcher.command_unacknowledge
lsst.sal.Watcher.command_unmute
lsst.sal.Watcher.logevent_alarm
lsst.sal.Watcher.logevent_appliedSettingsMatchStart
lsst.sal.Watcher.logevent_authList
lsst.sal.Watcher.logevent_configurationApplied
lsst.sal.Watcher.logevent_configurationsAvailable
lsst.sal.Watcher.logevent_errorCode
lsst.sal.Watcher.logevent_heartbeat
lsst.sal.Watcher.logevent_logLevel
lsst.sal.Watcher.logevent_logMessage
lsst.sal.Watcher.logevent_settingVersions
lsst.sal.Watcher.logevent_settingsApplied
lsst.sal.Watcher.logevent_simulationMode
lsst.sal.Watcher.logevent_softwareVersions
lsst.sal.Watcher.logevent_summary

In [4]:
# get all fields related to the Watcher mute
await client.get_fields('lsst.sal.Watcher.command_mute')

['WatcherID',
 'duration',
 'mutedBy',
 'name',
 'private_efdStamp',
 'private_host',
 'private_identity',
 'private_kafkaStamp',
 'private_origin',
 'private_rcvStamp',
 'private_revCode',
 'private_seqNum',
 'private_sndStamp',
 'severity']

In [5]:
# Get the duration data from October 24, 2024, to December 11, 2024.
start = Time("2024-10-24T00:00:00Z", scale="utc")
end = Time("2024-12-11T00:00:00Z", scale="utc")

mute_duration = await client.select_time_series(
                         "lsst.sal.Watcher.command_mute", 
                         ["WatcherID", '"duration"'], 
                          start, 
                          end
)

In [6]:
mute_duration

,WatcherID,duration
2024-10-24 03:40:56.166634+00:00,None,900
2024-10-24 03:41:00.434387+00:00,None,900
2024-10-24 13:54:53.129714+00:00,None,7200
2024-10-24 15:13:04.397697+00:00,None,604800
2024-10-24 15:42:58.214574+00:00,None,600
...,...,...
2024-12-10 19:07:21.871144+00:00,None,1800
2024-12-10 19:07:59.918454+00:00,None,1800
2024-12-10 23:18:27.236076+00:00,None,7200
2024-12-10 23:18:36.165743+00:00,None,7200


# Watcher Mute Duration: Histogram and Box Plot

In [7]:
# Number of Watcher
num_watcher = len(durations)
print(f"Number of watcher: {num_watcher}")

NameError: name 'durations' is not defined

In [ ]:
durations = mute_duration['duration']

# Histogram
plt.figure(figsize=(10, 5))
plt.hist(durations, bins=20, edgecolor="black", alpha=0.7)

# Plot
plt.xlabel("Mute Duration (seconds)")
plt.ylabel("Frequency")
plt.title("Histogram of Mute Durations (Watcher)")
plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.show()

There are some cases where the alarms are silenced much longer than in the rest, which makes analysis with this histogram difficult.

Let's convert the duration to hours.

In [ ]:
durations = mute_duration['duration']
durations_minutes = durations / 60  
durations_hours = durations_minutes / 60  

# Histogram
plt.figure(figsize=(10, 5))
plt.hist(durations_hours, bins=20, edgecolor="black", alpha=0.7)

# Plot
plt.xlabel("Mute Duration (hours)")
plt.ylabel("Frequency")
plt.title("Histogram of Mute Durations (Watcher)")
plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.show()

There's a case where the watcher has been muted for nearly 2,500 hours. What does this mean? Has the watcher gone several days without turning on?

Let's try to make a box plot

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=durations, color="skyblue")

plt.xlabel("Mute Duration (seconds)")
plt.title("Boxplot of Mute Durations (Watcher)")
plt.grid(axis="x", linestyle="--", alpha=0.7)

plt.show()

We are going to remove the outliers in order to study the duration of the muted watcher in most cases.

In [ ]:
Q1 = np.percentile(durations, 25)
Q3 = np.percentile(durations, 75)
IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

filtered_durations = [x for x in durations if x <= upper_bound]

# Histogram
plt.figure(figsize=(10, 5))
plt.hist(filtered_durations, bins=20, edgecolor="black", alpha=0.7)

plt.xlabel("Mute Duration (seconds)")
plt.ylabel("Frequency")
plt.title("Histogram of Mute Durations (without outliers)")
plt.grid(alpha=0.3)

plt.show()


There are still some cases that last a long time, making it difficult to see most of the chaos. But we can see that there are 40 cases with a duration of about 600,000 seconds (about 166 hours, about 7 days with the alarms muted). We also see a little over 20 alarms with just over 400,000 seconds, which would be about 5 days with the muted alarms.

If we calculate the mode of the duration of the watchers having eliminated the outliers but maintaining these muted watchers of 7 and 5 days we have the following analysis

In [ ]:
mode_result = stats.mode(filtered_durations, keepdims=True)
mode_value = mode_result.mode[0]

print(f"The duration of mode (without outliers): {mode_value:.2f} seconds")
print(f"The duration of mode (without outliers): {mode_value/60.:.2f} minutes")

We are going to select the watcher that are muted for less than a day (86400 seconds)

In [ ]:
upper_limit = 86400 # Seconds in a day
watchers_sel = [x for x in durations if x <= upper_limit]

num_watchers_selec = len(watchers_sel)
print(f"Number of watcher muted less than a day: {num_watchers_selec} of {num_watcher}")

# Histogram
plt.figure(figsize=(10, 5))
plt.hist(alarms_sel, bins=20, edgecolor="black", alpha=0.7)

plt.xlabel("Mute Duration (seconds)")
plt.ylabel("Frequency")
plt.title("Histogram of Mute Durations (without outliers)")
plt.grid(alpha=0.3)

plt.show()
